<a href="https://colab.research.google.com/github/philipmikh/CS1ReviewForCS2/blob/master/Final_Emotion_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -y update
!apt-get -y install ffmpeg
!pip -q install gradio faster-whisper soundfile torch sentence-transformers scikit-learn numpy pandas

import os
import re
import pickle
import tempfile
import numpy as np
import pandas as pd
import gradio as gr
import soundfile as sf

from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,786 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,66

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
CENTROIDS_PATH = "/content/drive/MyDrive/CLASS/emotion_project/emotion_avg.pkl"

if not os.path.exists(CENTROIDS_PATH):
    raise FileNotFoundError(
        f"Could not find {CENTROIDS_PATH}. Upload emotion_avg.pkl or fix the path."
    )

with open(CENTROIDS_PATH, "rb") as f:
    emotion_avg = pickle.load(f)

for k in list(emotion_avg.keys()):
    emotion_avg[k] = np.array(emotion_avg[k])

EMOTIONS = list(emotion_avg.keys())

print("Loaded emotions:", EMOTIONS)
print("Count:", len(EMOTIONS))


Loaded emotions: ['happy', 'grateful', 'lucky', 'upset', 'excited', 'nervous', 'proud', 'blessed', 'stressed', 'frustrated', 'sorry', 'tired', 'lonely', 'depressed', 'angry', 'overwhelmed', 'nauseous', 'emotional', 'awful', 'confident', 'anxious', 'lost', 'great', 'optimistic', 'guilty', 'terrible', 'uncomfortable', 'motivated', 'positive', 'insecure', 'hopeful', 'sad', 'nostalgic', 'inspired', 'low', 'love', 'exhausted', 'bad', 'sick', 'stupid', 'scared', 'good', 'discouraged', 'cold', 'down', 'helpless', 'hopeless', 'drained']
Count: 48


In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

embedder = SentenceTransformer(EMBED_MODEL_NAME)

whisper_model = WhisperModel(
    "base",
    compute_type="int8"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
SENT_END_RE = re.compile(r"(.+?[.!?]+)(\s+|$)", re.DOTALL)

In [ ]:
def predict_emotion_sentence(sentence):

    emb = embedder.encode([sentence], convert_to_numpy=True)[0]

    sims = []
    labels = []

    for emotion in EMOTIONS:
        centroid = emotion_avg[emotion]

        sim = cosine_similarity(
            emb.reshape(1, -1),
            centroid.reshape(1, -1)
        )[0][0]

        sims.append(sim)
        labels.append(emotion)

    order = np.argsort(sims)[::-1]

    best = order[0]
    second = order[1] if len(order) > 1 else order[0]

    return {
        "emotion": labels[best],
        "score": float(sims[best]),
        "margin": float(sims[best] - sims[second])
    }

In [ ]:
def transcribe_stream_and_label(audio):

    if audio is None:
        return "", "", 0.0, pd.DataFrame()

    audio_path = audio

    segments, _ = whisper_model.transcribe(audio_path)

    transcript_parts = []
    rows = []

    for seg in segments:

        text = seg.text.strip()

        if not text:
            continue

        transcript_parts.append(text)

        pred = predict_emotion_sentence(text)

        rows.append({
            "sentence": text,
            "emotion": pred["emotion"],
            "score": pred["score"],
            "margin": pred["margin"]
        })

    transcript = " ".join(transcript_parts)

    df = pd.DataFrame(rows)

    if len(rows) > 0:
        latest = rows[-1]
        latest_emotion = latest["emotion"]
        latest_margin = latest["margin"]
    else:
        latest_emotion = ""
        latest_margin = 0.0

    return transcript, latest_emotion, latest_margin, df

In [ ]:
with gr.Blocks(title="Emotion Speech Analyzer") as demo:

    gr.Markdown("# Emotion Speech Analyzer")

    with gr.Row():

        with gr.Column(scale=1):

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Audio Input"
            )

            run_btn = gr.Button("Analyze Audio")

        with gr.Column(scale=2):

            transcript_box = gr.Textbox(
                label="Transcript",
                lines=8
            )

            with gr.Row():

                latest_emotion = gr.Textbox(
                    label="Latest Emotion"
                )

                margin_box = gr.Number(
                    label="Match Margin"
                )

    results_table = gr.Dataframe(
        headers=["sentence", "emotion", "score", "margin"],
        label="Sentence Analysis"
    )

    run_btn.click(
        fn=transcribe_stream_and_label,
        inputs=audio_input,
        outputs=[
            transcript_box,
            latest_emotion,
            margin_box,
            results_table
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bfac8e04337ffcdf5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
